## Лаба 1

In [1]:
import pandas as pd #подключаем библиотеку pandas

In [2]:
df = pd.read_csv("BikeData.csv") #открываем файл

### 1 задание
Создайте новую переменную Total_count, в которую записать общее количество 
арендованных велосипедов за каждый час.

In [3]:
df['Total_count'] = df['Partner 1'] + df['Partner 2']

### 2 задание
Удалите ряды Partner 1 и Partner 2.

In [4]:
df = df.drop(columns= ['Partner 1', 'Partner 2'])

### 3 задание
Преобразуйте переменную Date к стандартной форме (to_datetime). 

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
#Функция библиотеки Pandas для преобразования в формат даты/времени(умеет распознавать различные форматы дат)

print(df['Date'].dtype) #dtype - Показывает тип данных, хранящихся в этом столбце

datetime64[us]


In [ ]:
print(df['Date'].dt.day.head(3))
#.dt — это специальный "аксессор" (доступ к свойствам даты и времени) в pandas. 
# Он используется для столбцов, содержащих даты/время (тип datetime64). 
# Он дает доступ к компонентам даты, таким как год, месяц, день, час и т.д.

0    1
1    1
2    1
Name: Date, dtype: int32


### 4 задание
Определите сколько велосипедов было выдано в день, месяц и число которого 
совпадает с днем выполнения этого задания. Определите, какой это был день недели.

In [ ]:
today = pd.Timestamp.today() #с помощью класса Timestamp я использую метод today, который показывает(возвращает новый объект из Timestamp) сегодняшнюю дату)
same_day = df[(df["Date"].dt.day == today.day) & (df["Date"].dt.month == today.month)]

In [8]:
total = same_day["Total_count"].sum()
print("Количество велосипедов:", total)

Количество велосипедов: 14319


In [ ]:
day_of_week = today.day_name(locale='ru_RU') 
#Метод Pandas для получения названия дня недели
#Возвращает строку с названием дня (Понедельник, Вторник и т.д.)

print("День недели:", day_of_week)

День недели: Пятница


In [66]:
s = df[(df["Date"].dt.day_name() == 'Monday') & (df["Temperature"] > 20)]
f = s.groupby("Date").count()
f.shape[0]

26

In [67]:
#s = df[(df["Date"].dt.day_name() == 'Monday') & day_name() == 'Monday')]
s = df[(df["Date"].dt.day_name() == 'Monday')]
res = s.groupby(s["Date"].dt.date)["Temperature"].max()
res = res[res > 20].count()
print(res)

26


In [ ]:
df["Temperature"] > 20

0       False
1       False
2       False
3       False
4       False
        ...  
8755    False
8756    False
8757    False
8758    False
8759    False
Name: Date, Length: 8760, dtype: bool

### 5 задание
Перекодируйте категориальную переменную переменную Functioning Day(рабочий или нерабочий день) в переменную типа boolean, т.е. "Yes" = True и "No" = False. Выведите количество нерабочих дней (не часов). День считается нерабочим, если пункт выдачи велосипедов работал менее 12 часов. 

In [ ]:
df["Functioning Day"] = df["Functioning Day"].map({"Yes": 1, "No": 0})#Заменяем Yes на 1, No на 0

In [11]:
df["Date"] = pd.to_datetime(df["Date"])

In [12]:
hours = df.groupby(df["Date"].dt.date)["Functioning Day"].sum()

In [13]:
non_working_days = (hours < 12).sum()
print("Количество нерабочих дней:", non_working_days)

Количество нерабочих дней: 12


### 6 задание
Перекодируйте переменную Holiday (являлся ли день праздничным) в 0, если No Holiday, и 1, если Holiday. (используйте анонимную функцию lambda). 

In [ ]:
df["Holiday"] = df["Holiday"].apply(lambda x: 1 if x == "Holiday" else 0) 
#apply - каждому элементу серии по отдельности
#означает, что функция принимает один аргумент, который мы называем x
#вернуть 1, если x равен строке "Holiday", в противном случае вернуть 0
#def f(x):
#    if x == "Holiday":
#        return 1
#    else:
#        return 0

### 7 задание
Введите новую категориальную переменную Temperature category, которая будет 
равна:
"Freezing", если температура < 0;
"Chilly", если 0 <= температура < 15;
"Nice", если 15 <= температура < 26;
"Hot", если 26 <= температура. 

In [ ]:
df["Temperature category"] = "Hot"
#df.loc[условие, "Temperature category"] - выбирает строки, где оба условия верны
df.loc[df["Temperature"] < 26, "Temperature category"] = "Nice"
df.loc[df["Temperature"] < 15, "Temperature category"] = "Chilly"
df.loc[df["Temperature"] < 0, "Temperature category"] = "Freezing"

df.loc[
    (df["Seasons"] == "Winter") & 
    (df["Temperature category"] == "Hot"),
    "Temperature category"
] = "Nice"

### 8 задание
Создайте новую переменную Good weather, которая будет равна 1 (т.е. хорошая погода), если:
Temperature category = "Nice", 
Humidity в диапазоне от 40 до 60 , 
скорость ветра Wind speed меньше 5.4, 
нет дождя (Rainfall)
нет снега (Snowfall). 
Определите, сколько процентов наблюдений соответствует хорошей погоде. 

In [ ]:
df["Good weather"] = (
    (df["Temperature category"] == "Nice") &
    (df["Humidity"].between(40, 60)) &
    (df["Wind speed"] < 5.4) &
    (df["Rainfall"] == 0) &
    (df["Snowfall"] == 0)
).astype(int) #1)Вычисляет логическое выражение (True/False) для каждой строки.2).astype(int) преобразует True в 1, False в 0

In [17]:
a = df["Good weather"].mean() * 100
print("Процент хорошей погоды:", a)

Процент хорошей погоды: 9.360730593607306


### 9 задание
Сгруппируйте данные по сезонам (Seasons), внутри сезона по категории погоды (Temperature category) и вывести для полученной группировки количество арендованных велосипедов. Вывести результаты в виде сгруппированной таблицы(через groupby) и в виде сводной таблицы (pivot_table).

In [ ]:
grouped = df.groupby(["Seasons", "Seasons category"])["Total_count"].sum()
print(grouped) 

Seasons  Temperature category
Autumn   Chilly                   775694
         Freezing                  12035
         Hot                      170674
         Nice                     811498
Spring   Chilly                   587211
         Freezing                   6331
         Hot                       86791
         Nice                     928572
Summer   Hot                     1347262
         Nice                     892664
Winter   Chilly                   215221
         Freezing                 258570
         Nice                       9536
Name: Total_count, dtype: int64


In [ ]:
pivot = pd.pivot_table(df,
    values="Total_count",
    index="Seasons",
    columns="Temperature category",
    aggfunc="sum"
)
#pd.pivot_table() - функция библиотеки Pandas для создания сводных таблиц

print(pivot)

Temperature category    Chilly  Freezing        Hot      Nice
Seasons                                                      
Autumn                775694.0   12035.0   170674.0  811498.0
Spring                587211.0    6331.0    86791.0  928572.0
Summer                     NaN       NaN  1347262.0  892664.0
Winter                215221.0  258570.0        NaN    9536.0


### 10 задание
Определите сезон и температуру, при которых выдается наибольшее и наименьшее количество велосипедов. 

In [ ]:
g = df.groupby(
    ["Seasons", "Temperature category"])["Total_count"].sum()

maxval = g.idxmax() 
#idxmax() - возвращает индекс (метку) элемента с максимальным значением в Series, для мультииндекса вернет кортеж вида (сезон, категория температуры), сохраняет этот индекс в переменную maxval
minval = g.idxmin() #idxmin() - возвращает индекс (метку) элемента с минимальным значением

print("Максимум:", maxval)
print("Минимум:", minval)


Максимум: ('Summer', 'Hot')
Минимум: ('Spring', 'Freezing')
